In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
from sklearn.cluster  import KMeans
from sklearn.preprocessing import StandardScaler


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df=pd.read_csv("customer_segmentation.csv")

In [ ]:
df.head()

In [ ]:
df.shape


In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()
#only the income column have null values 

In [ ]:
df.duplicated().sum()

In [ ]:
df["Income"] = df["Income"].fillna(df["Income"].median())

In [ ]:
df.isnull().sum()

In [ ]:
df["Age"] = 2026 - df["Year_Birth"]
#column for age 

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["Age"], bins=30, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.grid(True)
plt.show()

In [ ]:
#for income diostribution
plt.figure(figsize=(8,5))
sns.histplot(df["Income"], bins=30, kde=True)
plt.title("Income Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.grid(True)
plt.show()

In [ ]:
#graph for education count 
plt.figure(figsize=(8,5))
sns.countplot(x="Education", data=df)
plt.title("Education Level")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

In [ ]:
# martial status count 
plt.figure(figsize=(8,5))
sns.countplot(x="Marital_Status", data=df)
plt.title("Marital Status")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

In [ ]:
# We'll create new features that better represent customer behavior
df["Total_Spending"] = (
    df["MntWines"] +
    df["MntFruits"] +
    df["MntMeatProducts"] +
    df["MntFishProducts"] +
    df["MntSweetProducts"] +
    df["MntGoldProds"]
)

In [ ]:
#total children
df["Children"] = df["Kidhome"] + df["Teenhome"]

In [ ]:
#total purchases
df["Total_Purchases"] = (
    df["NumWebPurchases"] +
    df["NumCatalogPurchases"] +
    df["NumStorePurchases"]
)

In [ ]:
df[["Income", "Age", "Children", "Total_Spending", "Total_Purchases"]].head()

In [ ]:
features = [
    "Age",
    "Income",
    "Total_Spending",
    "Total_Purchases"
]

X = df[features]

In [ ]:
X.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled.shape

In [ ]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1,11), wcss, marker="o")

plt.title("Elbow Method")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.grid(True)

plt.show()

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)

df["Cluster"] = kmeans.fit_predict(X_scaled)

In [ ]:
df["Cluster"].value_counts().sort_index()

In [ ]:
centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=features
)

centers

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x="Income",
    y="Total_Spending",
    hue="Cluster",
    palette="Set2",
    s=80
)

plt.scatter(
    centers["Income"],
    centers["Total_Spending"],
    c="red",
    s=300,
    marker="X",
    label="Centroids"
)

plt.title("Customer Segmentation using K-Means", fontsize=16)
plt.xlabel("Income")
plt.ylabel("Total Spending")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
Q1 = df["Income"].quantile(0.25)
Q3 = df["Income"].quantile(0.75)

IQR = Q3 - Q1

df_plot = df[
    (df["Income"] >= Q1 - 1.5 * IQR) &
    (df["Income"] <= Q3 + 1.5 * IQR)
]

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df_plot,
    x="Income",
    y="Total_Spending",
    hue="Cluster",
    palette="Set2",
    s=80
)

plt.scatter(
    centers["Income"],
    centers["Total_Spending"],
    color="red",
    marker="X",
    s=300,
    label="Centroids"
)

plt.title("Customer Segmentation using K-Means", fontsize=16)
plt.xlabel("Income", fontsize=12)
plt.ylabel("Total Spending", fontsize=12)
plt.grid(True)
plt.legend()

plt.show()

In [ ]:
cluster_summary = df.groupby("Cluster")[[
    "Age",
    "Income",
    "Total_Spending",
    "Total_Purchases"
]].mean().round(2)

cluster_summary

In [ ]:
plt.figure(figsize=(7,5))

sns.countplot(data=df, x="Cluster", palette="Set2")

plt.title("Number of Customers in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Customers")
plt.grid(axis="y")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(data=df, x="Cluster", y="Income", palette="Set2")

plt.title("Income Distribution by Cluster")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(data=df, x="Cluster", y="Total_Spending", palette="Set2")

plt.title("Total Spending by Cluster")
plt.grid(True)

plt.show()

In [ ]:
import joblib

joblib.dump(kmeans, "customer_segmentation_model.pkl")
joblib.dump(scaler, "scaler.pkl")

In [ ]:
df.to_csv("customer_segments.csv", index=False)